# Content Approval Workflow

**Use Case**: End-to-end content creation and approval flow

1. Content Creator logs in
2. Content Creator creates content
3. Content Creator submits for approval
4. Admin logs in
5. Admin approves content
6. Content moves to repository

In [1]:
import requests
import json

# Service URLs
AUTH_URL = "http://localhost:8081"
TENANT_URL = "http://localhost:8082"
ROLE_PERM_URL = "http://localhost:8080"
NOTES_URL = "http://localhost:8088"
WORKFLOW_URL = "http://localhost:8086"

# Test tenant
TENANT_KEY = "euroschool"
TENANT_ID = None

# Users
content_creator = {
    "name": "Suresh",
    "email": "suresh@euroschool.com",
    "password": "Password123!"
}

admin = {
    "name": "Admin",
    "email": "admin@euroschool.com",
    "password": "Password123!"
}

# Storage
tokens = {}
user_ids = {}
role_ids = {}

def headers(token=None, tenant_id=None, user_id=None):
    h = {"Content-Type": "application/json"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    if tenant_id:
        h["X-Tenant-Id"] = str(tenant_id)
    if user_id:
        h["X-User-Id"] = str(user_id)
    return h

def print_response(resp, label="Response"):
    print(f"\n{label}: {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2))
    except:
        print(resp.text[:500] if resp.text else "(empty)")

print("Setup complete!")

Setup complete!


## Setup: Get Tenant & Create Roles/Users

One-time setup (skip if already done)

In [3]:
# Resolve existing tenant
resp = requests.get(f"{TENANT_URL}/v1/resolve", params={"tenantKey": TENANT_KEY})

if resp.status_code == 200:
    TENANT_ID = resp.json().get("tenantId")
    print(f"Found tenant: {TENANT_ID}")
else:
    # Create tenant
    payload = {"tenantKey": TENANT_KEY, "name": "Euroschool"}
    resp = requests.post(f"{TENANT_URL}/v1/tenants", json=payload, headers=headers())
    if resp.status_code in [200, 201]:
        TENANT_ID = resp.json().get("id")
        print(f"Created tenant: {TENANT_ID}")

print(f"TENANT_ID: {TENANT_ID}")

Found tenant: 0f66209d-f36f-4567-b24a-3d32a31bce80
TENANT_ID: 0f66209d-f36f-4567-b24a-3d32a31bce80


In [4]:
# Create roles (if not exists)
for role in [{"name": "CONTENT_CREATOR", "description": "Creates content"}, 
             {"name": "TENANT_ADMIN", "description": "Approves content"}]:
    payload = {"name": role["name"], "description": role["description"], "active": True}
    resp = requests.post(f"{ROLE_PERM_URL}/tenants/{TENANT_ID}/roles", json=payload, headers=headers(user_id="admin"))
    print_response(resp, f"Create Role: {role['name']}")
    if resp.status_code in [200, 201]:
        role_ids[role["name"]] = resp.json().get("id")

# Fetch all roles
resp = requests.get(f"{ROLE_PERM_URL}/tenants/{TENANT_ID}/roles")
if resp.status_code == 200:
    for r in resp.json():
        role_ids[r["name"]] = r["id"]

print(f"\nRole IDs: {role_ids}")


Create Role: CONTENT_CREATOR: 500
{
  "timestamp": "2026-01-26T14:29:01.231Z",
  "status": 500,
  "error": "Internal Server Error",
  "path": "/tenants/0f66209d-f36f-4567-b24a-3d32a31bce80/roles"
}

Create Role: TENANT_ADMIN: 500
{
  "timestamp": "2026-01-26T14:29:01.250Z",
  "status": 500,
  "error": "Internal Server Error",
  "path": "/tenants/0f66209d-f36f-4567-b24a-3d32a31bce80/roles"
}

Role IDs: {'PRINCIPAL': 3, 'SUBJECT_TEACHER': 4, 'STUDENT': 5, 'CONTENT_CREATOR': 9, 'TENANT_ADMIN': 10}


In [5]:
# Create users (if not exists)
for key, user in [("content_creator", content_creator), ("admin", admin)]:
    payload = {
        "tenantId": str(TENANT_ID),
        "email": user["email"],
        "password": user["password"],
        "name": user["name"],
        "joinMethod": "SELF_SIGNUP"
    }
    resp = requests.post(f"{AUTH_URL}/auth/signup", json=payload, headers=headers())
    print_response(resp, f"Signup: {user['name']}")
    if resp.status_code in [200, 201]:
        user_ids[key] = resp.json().get("userId")

print(f"\nUser IDs: {user_ids}")


Signup: Suresh: 200
{
  "accessToken": "7a556341-c268-44ed-b2e4-59dd80a305d4",
  "expiresIn": 3600,
  "refreshToken": "520ad017-b2a4-4ed4-ba28-5ca4a27884ed",
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "userId": "3320c76d-a3dd-4fc7-80fc-e73915f63824"
}

Signup: Admin: 200
{
  "accessToken": "881de70f-e0da-4ec0-a081-63f836fe87cf",
  "expiresIn": 3600,
  "refreshToken": "b25e6283-ea08-4602-a81f-593f1678ff8a",
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "userId": "609713d7-8c31-4bbb-9083-816e56bb0b9d"
}

User IDs: {'content_creator': '3320c76d-a3dd-4fc7-80fc-e73915f63824', 'admin': '609713d7-8c31-4bbb-9083-816e56bb0b9d'}


In [6]:
# Assign roles
for user_key, role_name in [("content_creator", "CONTENT_CREATOR"), ("admin", "TENANT_ADMIN")]:
    if user_key not in user_ids or role_name not in role_ids:
        print(f"Skipping {user_key} -> {role_name}")
        continue
    
    payload = {
        "roleId": str(role_ids[role_name]),
        "scopeType": "TENANT",
        "status": "ACTIVE"
    }
    resp = requests.post(
        f"{ROLE_PERM_URL}/tenants/{TENANT_ID}/users/{user_ids[user_key]}/roles",
        json=payload,
        headers=headers(user_id="admin")
    )
    print_response(resp, f"Assign {role_name} to {user_key}")


Assign CONTENT_CREATOR to content_creator: 200
{
  "id": 15,
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "userId": "3320c76d-a3dd-4fc7-80fc-e73915f63824",
  "roleId": 9,
  "scopeType": "TENANT",
  "scopeId": null,
  "status": "ACTIVE",
  "validFrom": null,
  "validTo": null,
  "assignedBy": "admin",
  "createdAt": "2026-01-26T14:29:07.919045Z"
}

Assign TENANT_ADMIN to admin: 200
{
  "id": 16,
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "userId": "609713d7-8c31-4bbb-9083-816e56bb0b9d",
  "roleId": 10,
  "scopeType": "TENANT",
  "scopeId": null,
  "status": "ACTIVE",
  "validFrom": null,
  "validTo": null,
  "assignedBy": "admin",
  "createdAt": "2026-01-26T14:29:07.932115Z"
}


---

# USE CASE BEGINS HERE

---

## Step 1: Content Creator Logs In

In [7]:
# Content Creator (Suresh) logs in
login_payload = {
    "tenantId": str(TENANT_ID),
    "identifier": content_creator["email"],
    "password": content_creator["password"]
}

resp = requests.post(f"{AUTH_URL}/auth/login", json=login_payload, headers=headers())
print_response(resp, "Content Creator Login")

if resp.status_code == 200:
    tokens["content_creator"] = resp.json().get("accessToken")
    user_ids["content_creator"] = resp.json().get("userId") or user_ids.get("content_creator")
    print(f"\n✓ Suresh logged in successfully")


Content Creator Login: 200
{
  "accessToken": "3731ac42-2b4a-4346-83a6-80bc85c4625b",
  "expiresIn": 3600,
  "refreshToken": "33c559bc-fde7-4ef0-afa3-6f7efb966059",
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "userId": "3568b7c6-21c0-4426-9830-a9406012d9c4"
}

✓ Suresh logged in successfully


## Step 2: Content Creator Creates Content

Suresh creates a note on "Plant Cell Structure"

In [8]:
note_payload = {
    "tenantId": str(TENANT_ID),
    "title": "Plant Cell Structure",
    "summary": "Detailed notes on plant cell components for Grade 4 Science",
    "contentMd": """# Plant Cell Structure

A plant cell has several important parts:

## 1. Cell Wall
- Rigid outer layer
- Made of cellulose
- Provides structure and support

## 2. Cell Membrane
- Controls what enters/exits cell
- Semi-permeable

## 3. Chloroplast
- Contains chlorophyll (green pigment)
- Site of photosynthesis
- Formula: $6CO_2 + 6H_2O \\rightarrow C_6H_{12}O_6 + 6O_2$

## 4. Vacuole
- Large, central
- Stores water and nutrients

## 5. Nucleus
- Control center of cell
- Contains DNA
""",
    "changeSummary": "Initial draft",
    "tags": ["#subject-science", "#topic-biology", "#grade-4", "#repository"],
    "scopeType": "TENANT",
    "createdBy": str(user_ids.get("content_creator"))
}

resp = requests.post(
    f"{NOTES_URL}/notes",
    json=note_payload,
    headers=headers(
        token=tokens.get("content_creator"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("content_creator")
    )
)
print_response(resp, "Create Note")

note_id = None
note_version_id = None
if resp.status_code in [200, 201]:
    note_id = resp.json().get("id")
    note_version_id = resp.json().get("latestVersionId")
    print(f"\n✓ Note created")
    print(f"  Note ID: {note_id}")
    print(f"  Version ID: {note_version_id}")


Create Note: 201
{
  "archiveReason": null,
  "archivedAt": null,
  "archivedBy": null,
  "createdAt": "2026-01-26T14:29:14.409666Z",
  "createdBy": "3568b7c6-21c0-4426-9830-a9406012d9c4",
  "deleted": false,
  "id": "f205eff3-f961-487a-92a8-a91de7eda1b2",
  "latestReleasedVersionId": null,
  "latestVersionId": "af72fe7e-c1b9-4a86-9c1a-464c6e49b304",
  "rejectionReason": null,
  "releasedAt": null,
  "releasedBy": null,
  "reviewedAt": null,
  "reviewedBy": null,
  "scopeId": null,
  "scopeType": "TENANT",
  "status": "DRAFT",
  "submittedAt": null,
  "submittedBy": null,
  "summary": "Detailed notes on plant cell components for Grade 4 Science",
  "tags": [
    "#grade-4",
    "#repository",
    "#subject-science",
    "#topic-biology"
  ],
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "title": "Plant Cell Structure",
  "updatedAt": "2026-01-26T14:29:14.409666Z",
  "updatedBy": "3568b7c6-21c0-4426-9830-a9406012d9c4"
}

✓ Note created
  Note ID: f205eff3-f961-487a-92a8-a91de

## Step 3: Content Creator Submits for Approval

Suresh creates a workflow and submits it for admin review

In [9]:
# First, create a workflow for this note
# Valid scopes: TENANT, CLASS, SECTION, GRADE, USER
# For repository-wide content, use TENANT scope
workflow_payload = {
    "contentId": str(note_id),
    "contentVersionId": str(note_version_id),
    "titleSnapshot": "Plant Cell Structure",
    "publishTargets": {
        "scope": "TENANT",
        "classIds": []
    }
}

print(f"Note ID: {note_id}")
print(f"Version ID: {note_version_id}")

resp = requests.post(
    f"{WORKFLOW_URL}/workflow",
    json=workflow_payload,
    headers=headers(
        token=tokens.get("content_creator"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("content_creator")
    )
)
print_response(resp, "Create Workflow")

workflow_id = None
if resp.status_code in [200, 201]:
    workflow_id = resp.json().get("workflowId") or resp.json().get("id")
    print(f"\n✓ Workflow created: {workflow_id}")

Note ID: f205eff3-f961-487a-92a8-a91de7eda1b2
Version ID: af72fe7e-c1b9-4a86-9c1a-464c6e49b304

Create Workflow: 201
{
  "contentId": "f205eff3-f961-487a-92a8-a91de7eda1b2",
  "contentVersionId": "af72fe7e-c1b9-4a86-9c1a-464c6e49b304",
  "createdAt": "2026-01-26T14:29:19.410086Z",
  "createdBy": "3568b7c6-21c0-4426-9830-a9406012d9c4",
  "currentStep": null,
  "lastUpdatedAt": "2026-01-26T14:29:19.410086Z",
  "lastUpdatedBy": "3568b7c6-21c0-4426-9830-a9406012d9c4",
  "publishTargets": {
    "classIds": [],
    "gradeIds": null,
    "scope": "TENANT",
    "sectionIds": null,
    "userIds": null
  },
  "requiredApprovals": null,
  "state": "DRAFT",
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "titleSnapshot": "Plant Cell Structure",
  "version": 0,
  "workflowId": "3718b509-35bc-4826-afe2-982f9176cd10"
}

✓ Workflow created: 3718b509-35bc-4826-afe2-982f9176cd10


In [ ]:
# Submit for review - assign Admin as reviewer
submit_payload = {
    "reviewerUserIds": [str(user_ids.get("admin"))],
    "requiredApprovals": 1,
    "note": "Please review this Plant Cell Structure content for Grade 4 Science."
}

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/{workflow_id}/submit",
    json=submit_payload,
    headers=headers(
        token=tokens.get("content_creator"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("content_creator")
    )
)
print_response(resp, "Submit for Review")

review_task_id = None
if resp.status_code in [200, 201]:
    data = resp.json()
    # Extract review task ID from reviewTasks array
    review_tasks = data.get("reviewTasks") or []
    if review_tasks:
        review_task_id = review_tasks[0].get("id")
    print(f"\n✓ Submitted for review")
    print(f"  Workflow State: {data.get('state')}")
    print(f"  Review Task ID: {review_task_id}")

---

## Step 4: Admin Logs In

In [11]:
# Admin logs in
login_payload = {
    "tenantId": str(TENANT_ID),
    "identifier": admin["email"],
    "password": admin["password"]
}

resp = requests.post(f"{AUTH_URL}/auth/login", json=login_payload, headers=headers())
print_response(resp, "Admin Login")

if resp.status_code == 200:
    tokens["admin"] = resp.json().get("accessToken")
    user_ids["admin"] = resp.json().get("userId") or user_ids.get("admin")
    print(f"\n✓ Admin logged in successfully")


Admin Login: 200
{
  "accessToken": "35d8ef01-335e-4152-886b-a8734656d806",
  "expiresIn": 3600,
  "refreshToken": "e19be5c0-4aa7-417a-b330-fbbf8c1489c7",
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "userId": "10f12461-199b-488f-bc25-be3325735d5f"
}

✓ Admin logged in successfully


## Step 5: Admin Approves Content

Admin views pending reviews and approves the content

In [12]:
# Admin views workflow details
resp = requests.get(
    f"{WORKFLOW_URL}/workflow/{workflow_id}",
    headers=headers(
        token=tokens.get("admin"),
        tenant_id=TENANT_ID
    )
)
print_response(resp, "Admin Views Workflow")

# Extract review_task_id if not already set
if not review_task_id and resp.status_code == 200:
    workflow_data = resp.json()
    reviews = workflow_data.get("reviews") or workflow_data.get("reviewTasks") or []
    if reviews:
        review_task_id = reviews[0].get("id")
        print(f"\nReview Task ID: {review_task_id}")


Admin Views Workflow: 200
{
  "contentId": "f205eff3-f961-487a-92a8-a91de7eda1b2",
  "contentVersionId": "af72fe7e-c1b9-4a86-9c1a-464c6e49b304",
  "createdAt": "2026-01-26T14:29:19.410086Z",
  "createdBy": "3568b7c6-21c0-4426-9830-a9406012d9c4",
  "currentStep": "REVIEW",
  "lastUpdatedAt": "2026-01-26T14:29:24.903899Z",
  "lastUpdatedBy": "3568b7c6-21c0-4426-9830-a9406012d9c4",
  "publishTargets": {
    "classIds": [],
    "gradeIds": null,
    "scope": "TENANT",
    "sectionIds": null,
    "userIds": null
  },
  "requiredApprovals": 1,
  "state": "IN_REVIEW",
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "titleSnapshot": "Plant Cell Structure",
  "version": 1,
  "workflowId": "3718b509-35bc-4826-afe2-982f9176cd10"
}


In [13]:
# Admin approves the content
approve_payload = {
    "comment": "Content is accurate and appropriate for Grade 4. Approved."
}

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/{workflow_id}/reviews/{review_task_id}/approve",
    json=approve_payload,
    headers=headers(
        token=tokens.get("admin"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("admin")
    )
)
print_response(resp, "Admin Approves")

if resp.status_code in [200, 201]:
    print(f"\n✓ Content APPROVED by Admin")


Admin Approves: 400
{
  "error": "BAD_REQUEST",
  "message": "Invalid value for parameter taskId",
  "traceId": "c9c58872-2605-4405-a561-db662b5dd0f0"
}


## Step 6: Content Moves to Repository

After approval, publish the content to the repository

In [14]:
# Publish to repository (TENANT scope = available to all in tenant)
publish_payload = {
    "publishAt": None,  # Immediate
    "publishTargets": {
        "scope": "TENANT",
        "classIds": []
    }
}

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/{workflow_id}/publish",
    json=publish_payload,
    headers=headers(
        token=tokens.get("admin"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("admin")
    )
)
print_response(resp, "Publish to Repository")

if resp.status_code in [200, 201]:
    print(f"\n✓ Content PUBLISHED to repository")


Publish to Repository: 409
{
  "error": "CONFLICT",
  "message": "Workflow is not approved for publish",
  "traceId": "048ef8bd-0fb9-48d1-a6e3-3f10d866d1f4"
}


## Verification: Check Final State

In [15]:
# Verify workflow is in PUBLISHED state
resp = requests.get(
    f"{WORKFLOW_URL}/workflow/{workflow_id}",
    headers=headers(
        token=tokens.get("admin"),
        tenant_id=TENANT_ID
    )
)
print_response(resp, "Final Workflow State")

if resp.status_code == 200:
    state = resp.json().get("state")
    print(f"\n========================================")
    print(f"  WORKFLOW STATE: {state}")
    print(f"========================================")


Final Workflow State: 200
{
  "contentId": "f205eff3-f961-487a-92a8-a91de7eda1b2",
  "contentVersionId": "af72fe7e-c1b9-4a86-9c1a-464c6e49b304",
  "createdAt": "2026-01-26T14:29:19.410086Z",
  "createdBy": "3568b7c6-21c0-4426-9830-a9406012d9c4",
  "currentStep": "REVIEW",
  "lastUpdatedAt": "2026-01-26T14:29:24.903899Z",
  "lastUpdatedBy": "3568b7c6-21c0-4426-9830-a9406012d9c4",
  "publishTargets": {
    "classIds": [],
    "gradeIds": null,
    "scope": "TENANT",
    "sectionIds": null,
    "userIds": null
  },
  "requiredApprovals": 1,
  "state": "IN_REVIEW",
  "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
  "titleSnapshot": "Plant Cell Structure",
  "version": 1,
  "workflowId": "3718b509-35bc-4826-afe2-982f9176cd10"
}

  WORKFLOW STATE: IN_REVIEW


In [19]:
# Verify note is available in repository (search by tag)
# Note: Use URL-encoded tag or without # prefix
resp = requests.get(
    f"{NOTES_URL}/notes",
    params={"tag": "repository", "scopeType": "TENANT"},
    headers=headers(
        token=tokens.get("admin"),
        tenant_id=TENANT_ID
    )
)
print_response(resp, "Repository Contents")

if resp.status_code == 200:
    notes = resp.json() if isinstance(resp.json(), list) else resp.json().get("content", [])
    print(f"\n✓ Found {len(notes)} notes in repository")


Repository Contents: 200
{
  "items": [
    {
      "archiveReason": null,
      "archivedAt": null,
      "archivedBy": null,
      "createdAt": "2026-01-26T14:29:14.409666Z",
      "createdBy": "3568b7c6-21c0-4426-9830-a9406012d9c4",
      "deleted": false,
      "id": "f205eff3-f961-487a-92a8-a91de7eda1b2",
      "latestReleasedVersionId": null,
      "latestVersionId": "af72fe7e-c1b9-4a86-9c1a-464c6e49b304",
      "rejectionReason": null,
      "releasedAt": null,
      "releasedBy": null,
      "reviewedAt": null,
      "reviewedBy": null,
      "scopeId": null,
      "scopeType": "TENANT",
      "status": "DRAFT",
      "submittedAt": null,
      "submittedBy": null,
      "summary": "Detailed notes on plant cell components for Grade 4 Science",
      "tags": [
        "#grade-4",
        "#repository",
        "#subject-science",
        "#topic-biology"
      ],
      "tenantId": "0f66209d-f36f-4567-b24a-3d32a31bce80",
      "title": "Plant Cell Structure",
      "updatedAt": 

In [18]:
# Debug: Check workflow state and note state
print("=== Workflow State ===")
resp = requests.get(
    f"{WORKFLOW_URL}/workflow/{workflow_id}",
    headers=headers(token=tokens.get("admin"), tenant_id=TENANT_ID)
)
if resp.status_code == 200:
    wf = resp.json()
    print(f"Workflow ID: {workflow_id}")
    print(f"State: {wf.get('state')}")
else:
    print_response(resp, "Workflow")

print("\n=== Note State ===")
resp = requests.get(
    f"{NOTES_URL}/notes/{note_id}",
    headers=headers(token=tokens.get("admin"), tenant_id=TENANT_ID)
)
if resp.status_code == 200:
    note = resp.json()
    print(f"Note ID: {note_id}")
    print(f"Status: {note.get('status')}")
    print(f"Tags: {note.get('tags')}")
else:
    print_response(resp, "Note")

=== Workflow State ===
Workflow ID: 3718b509-35bc-4826-afe2-982f9176cd10
State: IN_REVIEW

=== Note State ===
Note ID: f205eff3-f961-487a-92a8-a91de7eda1b2
Status: DRAFT
Tags: ['#grade-4', '#repository', '#subject-science', '#topic-biology']


## Summary

```
┌─────────────────────────────────────────────────────────────┐
│                  CONTENT APPROVAL WORKFLOW                  │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. Content Creator (Suresh) logs in                        │
│                    │                                        │
│                    ▼                                        │
│  2. Creates Note: "Plant Cell Structure"                    │
│     Status: DRAFT                                           │
│                    │                                        │
│                    ▼                                        │
│  3. Submits for Approval                                    │
│     Status: IN_REVIEW                                       │
│                    │                                        │
│                    ▼                                        │
│  4. Admin logs in                                           │
│                    │                                        │
│                    ▼                                        │
│  5. Admin approves content                                  │
│     Status: APPROVED                                        │
│                    │                                        │
│                    ▼                                        │
│  6. Content published to Repository                         │
│     Status: PUBLISHED                                       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

## Cleanup

Run these cells to delete test data created during this workflow test.

In [20]:
# Cleanup: Delete the note created during this test
if note_id:
    resp = requests.delete(
        f"{NOTES_URL}/notes/{note_id}",
        headers=headers(
            token=tokens.get("admin"),
            tenant_id=TENANT_ID,
            user_id=user_ids.get("admin")
        )
    )
    if resp.status_code in [200, 204]:
        print(f"✓ Deleted note: {note_id}")
    else:
        print_response(resp, f"Delete Note {note_id}")
else:
    print("No note_id to delete")

✓ Deleted note: f205eff3-f961-487a-92a8-a91de7eda1b2


In [21]:
# Cleanup: Delete all "Plant Cell Structure" notes (from previous test runs)
resp = requests.get(
    f"{NOTES_URL}/notes",
    params={"q": "Plant Cell Structure", "scopeType": "TENANT"},
    headers=headers(token=tokens.get("admin"), tenant_id=TENANT_ID)
)

if resp.status_code == 200:
    data = resp.json()
    notes = data.get("items") or data.get("content") or (data if isinstance(data, list) else [])
    deleted_count = 0
    for note in notes:
        nid = note.get("id")
        if nid:
            del_resp = requests.delete(
                f"{NOTES_URL}/notes/{nid}",
                headers=headers(token=tokens.get("admin"), tenant_id=TENANT_ID, user_id=user_ids.get("admin"))
            )
            if del_resp.status_code in [200, 204]:
                deleted_count += 1
                print(f"✓ Deleted note: {nid}")
            else:
                print(f"✗ Failed to delete {nid}: {del_resp.status_code}")
    print(f"\nTotal deleted: {deleted_count} notes")
else:
    print_response(resp, "List Notes")

✓ Deleted note: a4827694-3139-458f-81a0-9d988227db7a

Total deleted: 1 notes


In [22]:
# Cleanup: Verify cleanup was successful
print("=== Cleanup Verification ===\n")

# Check remaining notes with repository tag
resp = requests.get(
    f"{NOTES_URL}/notes",
    params={"tag": "repository", "scopeType": "TENANT"},
    headers=headers(token=tokens.get("admin"), tenant_id=TENANT_ID)
)

if resp.status_code == 200:
    data = resp.json()
    notes = data.get("items") or data.get("content") or (data if isinstance(data, list) else [])
    print(f"Remaining notes with #repository tag: {len(notes)}")
    for note in notes:
        print(f"  - {note.get('id')}: {note.get('title')}")
else:
    print_response(resp, "List Notes")

print("\n✓ Cleanup complete!")

=== Cleanup Verification ===

Remaining notes with #repository tag: 0

✓ Cleanup complete!
